# 05 - Synthetic Control: Robustness Checks
**Owner:** Member 2 (Dhananjay) · **Week 3**

`03_synthetic_control.ipynb` estimated that North's price rise lifted revenue by about 3-4%. This notebook asks: **can we trust that number?** Each test tries to break the result. If the effect survives, it is more believable.

## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

df = pd.read_csv("project1_algorithmic_pricing_synthetic_dataset.csv", parse_dates=["date"])
daily = df.pivot(index="date", columns="region", values="revenue")
weekly = daily.resample("W-SUN").sum()
weekly = weekly[daily.resample("W-SUN").size() == 7]

TREATMENT_DATE = pd.Timestamp("2025-05-01")

def fit_weights(data, treated, donors, cutoff):
    pre = data.index < cutoff
    X, y = data.loc[pre, donors].values, data.loc[pre, treated].values
    scale = y.mean()
    res = minimize(lambda w: np.mean(((y - X @ w) / scale) ** 2),
                   np.full(len(donors), 1 / len(donors)),
                   bounds=[(0, 1)] * len(donors),
                   constraints={"type": "eq", "fun": lambda w: w.sum() - 1})
    return pd.Series(res.x, index=donors)

def lift(data, treated, donors, cutoff, end=None):
    """Fit on data before `cutoff`, return % gap between `cutoff` and `end`."""
    w = fit_weights(data, treated, donors, cutoff)
    synth = data[donors] @ w
    window = (data.index >= cutoff) & ((data.index < end) if end is not None else True)
    pre = data.index < cutoff
    pre_err = np.sqrt(((data.loc[pre, treated] - synth[pre]) ** 2).mean()) / data.loc[pre, treated].mean()
    effect = (data.loc[window, treated] - synth[window]).mean() / synth[window].mean()
    return effect * 100, pre_err * 100, w

main_lift, main_err, main_w = lift(weekly, "North", ["South", "West"], TREATMENT_DATE)
print(f"Main result (weekly): lift {main_lift:.1f}%, pre-period error {main_err:.1f}%")

Main result (weekly): lift 3.2%, pre-period error 4.0%


## Test 1 - Placebo in time
Pretend the price change happened on **1 March** instead of 1 May, using only data before the real change. Nothing actually happened in March, so a good method should find roughly **0%**.

In [2]:
placebo_date = pd.Timestamp("2025-03-01")
before_real = weekly[weekly.index < TREATMENT_DATE]
t1_lift, t1_err, _ = lift(before_real, "North", ["South", "West"], placebo_date)
print(f"Fake change on 1 March: 'lift' = {t1_lift:.1f}%  (real change: {main_lift:.1f}%)")

Fake change on 1 March: 'lift' = -1.8%  (real change: 3.2%)
